# Image Captioning using ViT + GPT-2

**Kaggle notebook · Flickr8k · Transfer learning**

This project uses a pretrained **Vision Transformer (ViT)** to extract image features, and a pretrained **GPT-2** decoder to generate captions. We connect them using **new cross-attention layers** and train the decoder on Flickr8k.

We do **not** train ViT or GPT-2 from scratch. The new image-to-text connection is trained here.

**Before you start:** Attach the [Flickr8k dataset](https://www.kaggle.com/datasets/adityajn105/flickr8k) using Kaggle **Add Input**, and enable **GPU** and **Internet** in notebook settings.

## 1. Install the missing packages
Run this once. If Kaggle asks you to restart the session, restart it before continuing.

In [ ]:
%pip -q install "transformers==4.57.6" "sacrebleu>=2.4,<3"

## 2. Imports and settings
All the values you may want to change are in this one cell. Start with the small run to check for GPU memory problems.

In [ ]:
import random
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer,
    GPT2Config,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Small first experiment. Increase later if training is stable.
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 100
TEST_SAMPLES = 100
EPOCHS = 3
BATCH_SIZE = 1            # Keep 1 if your Kaggle kernel crashes.
GRAD_ACCUM_STEPS = 4       # Accumulate gradients before updating weights.
MAX_LENGTH = 24
LEARNING_RATE = 5e-5
USE_AMP = (device == "cuda")  # Reduces GPU memory usage.

## 3. Load Flickr8k from Kaggle
No manual download is needed. This finds `captions.txt` next to the `Images` folder in your attached Kaggle dataset.

In [ ]:
folders = [
    p.parent for p in Path("/kaggle/input").rglob("captions.txt")
    if (p.parent / "Images").is_dir()
]

if not folders:
    raise FileNotFoundError(
        "Attach adityajn105/flickr8k using Add Input. "
        "We need captions.txt and an Images folder."
    )

dataset_folder = folders[0]
caption_file = dataset_folder / "captions.txt"
image_folder = dataset_folder / "Images"

df = pd.read_csv(caption_file)
df = df.dropna(subset=["image", "caption"])

print("Captions:", len(df))
print("Unique images:", df["image"].nunique())
print(df.head(2))

## 4. Split images into TRAIN / VALIDATION / TEST
Five captions belong to each image. We split **image names**, not caption rows, so the same photo never appears in two sets. This is a custom reproducible split, not the official Flickr8k benchmark split.

In [ ]:
captions_by_image = df.groupby("image")["caption"].apply(list)
image_names = [
    name for name in captions_by_image.index
    if len(captions_by_image[name]) >= 5 and (image_folder / name).is_file()
]

random.Random(SEED).shuffle(image_names)
train_end = int(0.8 * len(image_names))
val_end = int(0.9 * len(image_names))

train_names = image_names[:train_end][:TRAIN_SAMPLES]
val_names = image_names[train_end:val_end][:VAL_SAMPLES]
test_names = image_names[val_end:][:TEST_SAMPLES]

assert train_names and val_names and test_names, "Not enough images for the split."
assert not (set(train_names) & set(val_names))
assert not (set(train_names) & set(test_names))
assert not (set(val_names) & set(test_names))

print("Train:", len(train_names))
print("Validation:", len(val_names))
print("Test:", len(test_names))

## 5. Look at one training image and its five captions

In [ ]:
example_name = train_names[0]
with Image.open(image_folder / example_name) as img:
    display(img.convert("RGB"))

for text in captions_by_image[example_name][:5]:
    print("-", text)

## 6. Load and connect pretrained ViT and GPT-2
ViT learns visual representations. GPT-2 is configured as a decoder with **cross-attention**. The cross-attention weights show as **MISSING** because GPT-2 was not originally trained with images. That warning is expected.

In [ ]:
VIT_MODEL = "google/vit-base-patch16-224-in21k"
GPT2_MODEL = "gpt2"

image_processor = ViTImageProcessor.from_pretrained(VIT_MODEL)
tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Important: GPT-2 needs to be configured as a decoder.
decoder_config = GPT2Config.from_pretrained(GPT2_MODEL)
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True

model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    VIT_MODEL,
    GPT2_MODEL,
    decoder_config=decoder_config,
)

model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.decoder_start_token_id = tokenizer.bos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

# Keep pretrained ViT weights unchanged. Train GPT-2 + cross-attention.
for parameter in model.encoder.parameters():
    parameter.requires_grad = False

model = model.to(device)
print("Cross-attention enabled:", model.decoder.config.add_cross_attention)
print("Training GPT-2 with ViT frozen")

## 7. Prepare one mini-batch
Images are read **only when needed**, instead of loading the whole image dataset into GPU memory. For training, a random one of the five human-written captions is selected.

In [ ]:
def prepare_batch(names, training):
    images = []
    texts = []

    for name in names:
        with Image.open(image_folder / name) as img:
            images.append(img.convert("RGB"))

        caption = (
            random.choice(captions_by_image[name])
            if training else captions_by_image[name][0]
        )
        texts.append(str(caption).strip() + tokenizer.eos_token)

    pixels = image_processor(images=images, return_tensors="pt")
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = tokens["input_ids"].clone()
    labels[tokens["attention_mask"] == 0] = -100
    return pixels.pixel_values.to(device), labels.to(device)

## 8. TRAIN and VALIDATE
Only the training pass updates weights. Validation uses separate images and does not change weights. Mixed precision and batch size 1 help reduce GPU memory use.

In [ ]:
optimizer = torch.optim.AdamW(model.decoder.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(names, training):
    model.train() if training else model.eval()
    model.encoder.eval()  # ViT is frozen.

    order = list(names)
    if training:
        random.shuffle(order)
        optimizer.zero_grad(set_to_none=True)

    losses = []
    batch_starts = range(0, len(order), BATCH_SIZE)

    for step, start in enumerate(tqdm(batch_starts, leave=False), 1):
        batch_names = order[start:start + BATCH_SIZE]
        pixel_values, labels = prepare_batch(batch_names, training)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                loss = model(pixel_values=pixel_values, labels=labels).loss

        losses.append(loss.item())

        if training:
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

            final_batch = start + BATCH_SIZE >= len(order)
            if step % GRAD_ACCUM_STEPS == 0 or final_batch:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

    return sum(losses) / len(losses)


train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    train_loss = run_epoch(train_names, training=True)
    val_loss = run_epoch(val_names, training=False)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS}: train={train_loss:.4f}, val={val_loss:.4f}")

## 9. Plot training and validation loss
If training loss falls but validation loss rises, the model may be overfitting.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o", label="Train")
plt.plot(range(1, EPOCHS + 1), val_losses, marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ViT + GPT-2: Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Generate a caption from an image
This function works for a Flickr8k image or any image path attached to Kaggle.

In [ ]:
@torch.inference_mode()
def generate_caption(image_path):
    with Image.open(image_path) as img:
        image = img.convert("RGB")

    pixels = image_processor(
        images=image, return_tensors="pt"
    ).pixel_values.to(device)

    model.eval()
    ids = model.generate(
        pixel_values=pixels,
        max_new_tokens=24,
        num_beams=3,
    )
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

## 11. TEST — evaluate on unseen images
BLEU measures overlap with all **five reference captions per image**. This small test run is not directly comparable to published Flickr8k benchmarks because we use a custom split.

In [ ]:
predictions = []
for name in tqdm(test_names):
    predictions.append(generate_caption(image_folder / name))

references = [
    [str(captions_by_image[name][i]) for name in test_names]
    for i in range(5)
]

bleu = sacrebleu.corpus_bleu(predictions, references)
print("Test BLEU:", round(bleu.score, 2))

## 12. See predicted captions
A caption can use different words from the references and still be reasonable. Look at the image when judging correctness.

In [ ]:
for i in range(min(5, len(test_names))):
    name = test_names[i]
    with Image.open(image_folder / name) as img:
        display(img.convert("RGB"))

    print("Human captions:")
    for text in captions_by_image[name][:5]:
        print("-", text)
    print("Prediction:", predictions[i])
    print("-" * 50)

## 13. Try an image of your own (optional)
Add an image using Kaggle **Add Input**, then paste its full path below. Leave the path blank to reuse a test image.

In [ ]:
CUSTOM_IMAGE_PATH = ""  # Example: "/kaggle/input/my-images/photo.jpg"
image_path = CUSTOM_IMAGE_PATH or str(image_folder / test_names[0])

with Image.open(image_path) as img:
    display(img.convert("RGB"))
print("Generated caption:", generate_caption(image_path))

## 14. Save the trained model (optional)
Save to `/kaggle/working/` and use **Save Version** to preserve the result. Model files are large, so leave this disabled until you need a checkpoint.

In [ ]:
SAVE_MODEL = False

if SAVE_MODEL:
    output_dir = "/kaggle/working/vit_gpt2_custom"
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    image_processor.save_pretrained(output_dir)
    print("Model saved at:", output_dir)

## Simple explanation for interviews

> I used a pretrained ViT encoder to extract visual features and a pretrained GPT-2 decoder to generate captions. I connected them through newly initialized cross-attention layers, froze ViT, and trained the decoder on Flickr8k. I evaluated the model on separate images using BLEU and qualitative examples.

### Important limitations

- The new cross-attention layers need more training than an already-caption-trained checkpoint.
- Running with 1,000 images is a quick experiment, not a strong benchmark.
- If Kaggle runs out of memory, keep `BATCH_SIZE = 1`, use GPU and mixed precision, and lower the sample count for your first successful run.